# Script to help generate automatically the right parameter.yaml files

In [1]:
import sys
import yaml
import json
import pprint
import copy
from itertools import product


In [2]:
original_filename = "../param.yaml"
original_params = yaml.safe_load(open(original_filename))

In [3]:
pprint.pprint(original_params)

{'comparison_arguments': {'compared_text_type': 'steered_bis'},
 'data_arguments': {'input_version': 'guardian_from_nov2025_articles_v2.10k',
                    'max_loaded_samples': 1000,
                    'truncate_input_words': 6},
 'detection_arguments': {'lda_parameters': {'shrinkage': None,
                                            'solver': 'lsqr'},
                         'max_seq_length': 512,
                         'mlp_parameters': {'batch_size': 512,
                                            'hidden_dims': [2048, 64, 64, 32],
                                            'learning_rate': 0.001,
                                            'num_epochs': 1},
                         'model_type': 'mlp',
                         'number_of_bits': 3,
                         'number_prompts_truncation': None,
                         'seed': 1,
                         'sentence_array': False,
                         'token_aggregation': False,
                         

## Parameter hard set

In [9]:
variant_params = copy.deepcopy(original_params)

variant_params["data_arguments"]['max_loaded_samples'] = 1000
variant_params['steering_arguments']["noise_offset"] = 1
variant_params['steering_arguments']["noise_type"] = "sparse_0.003"
variant_params["detection_arguments"]['number_of_bits'] = 2
variant_params["gathering_arguments"]['gathering_layers'] = [14]
variant_params["steering_arguments"]["steering_layers"] = [14]


output_param_name = "Small_llm_rerun" #"traditional_watermarking" #"main_table_m5" #"bigger_set_quality_check"
output_root_filename = "../params/" + output_param_name + "_"

python_script = "src/multibit_pipeline.py"
#"src/classic_watermarking.py"

variant_params["run_name"] = output_param_name



## Parameter dynamic crossing

In [10]:
variantion_dict = {
    "model_arguments": {
        'model_id': [
            # 'meta-llama/Llama-3.1-8B', 
            'meta-llama/Llama-3.2-3B', 
            # 'meta-llama/Llama-3.2-1B',
            # 'mistralai/Ministral-3-8B'
        ],
    },
    'data_arguments': {
        'input_version': [
            'guardian_from_nov2025_articles_v2.10k', 
            'Hello-SimpleAI/HC3',
        ],
    },
    'steering_arguments': {
    #     "noise_type": [
    #         "sparse_0.003",
    # #         "uniform",
    #     ],
        "noise_offset": [1, 2, 3, 4, 5],
        "noise_max": [5],
    #     "noise_max": [0.05, 0.1, 0.15, 0.2],
    },
    # "robustness_arguments": {
    #     "paraphrasing": {
    #         "order_diversity": [20, 40, 60]
    #     }
    # }
}

In [11]:
def conditional_cast(this_dic, value):
    assert isinstance(this_dic, dict)
    # Adjust parameters conditionally based on other parameter values
    if value == "guardian_from_nov2025_articles_v1.10k":
        this_dic['generation_arguments']['generation_batch_size'] = 64
        if this_dic['model_arguments']['model_id'] == 'mistralai/Ministral-3-8B':
            this_dic['model_arguments']['model_id'] = 'mistralai/Ministral-3-8B-Base-2512'

    elif value == "Hello-SimpleAI/HC3":
        this_dic['generation_arguments']['generation_batch_size'] = 32
        if this_dic['model_arguments']['model_id'] == 'mistralai/Ministral-3-8B':
            this_dic['model_arguments']['model_id'] = 'mistralai/Ministral-3-8B-Instruct-2512'
        else:
            this_dic['model_arguments']['model_id'] += '-Instruct'

In [12]:
def generate_parameter_variants(base_params, variation_dict):
    """
    Generate all combinations of parameters from a variation dictionary.
    
    Args:
        base_params: Base parameter dictionary to modify
        variation_dict: Dictionary with same structure as base_params, 
                       but with lists of values to try for each key
    
    Returns:
        List of parameter dictionaries, one for each combination
    """
    # Extract all paths and their corresponding value lists
    paths_and_values = []
    
    def extract_paths(d, current_path=[]):
        """Recursively extract all paths and their value lists"""
        for key, value in d.items():
            new_path = current_path + [key]
            if isinstance(value, dict):
                extract_paths(value, new_path)
            elif isinstance(value, list):
                paths_and_values.append((new_path, value))
    
    extract_paths(variation_dict)
    
    # Generate all combinations
    paths = [p[0] for p in paths_and_values]
    value_lists = [p[1] for p in paths_and_values]
    combinations = list(product(*value_lists))

    print("combinations", combinations)
    print("Number of combinations:", len(combinations))
    print("Paths:", paths)

    # Create a parameter dict for each combination
    variants = []
    output_filenames = []
    for combo in combinations:
        # Start with a deep copy of base params
        variant = copy.deepcopy(base_params)
        
        # Apply each value in the combination
        for path, value in zip(paths, combo):
            # Navigate to the correct nested location
            current = variant
            for key in path[:-1]:
                if key not in current:
                    current[key] = {}
                current = current[key]
            # Set the value
            current[path[-1]] = value

        for value in combo:
            conditional_cast(variant, value)
        
        variants.append(variant)

        # Save the variant to a YAML file
        output_filename = f"{output_root_filename}"
        for value in combo:
            safe_value = str(value).replace('/', '_').replace(' ', '_')
            output_filename += f"_{safe_value}"
        output_filename += ".yaml"
        print("Saving variant to:", output_filename)
        with open(output_filename, 'w') as f:
            yaml.dump(variant, f)
        output_filenames.append(output_filename)
    
    return variants, output_filenames


In [13]:
param_list, file_names = generate_parameter_variants(variant_params, variantion_dict)

combinations [('meta-llama/Llama-3.2-3B', 'guardian_from_nov2025_articles_v2.10k', 1, 5), ('meta-llama/Llama-3.2-3B', 'guardian_from_nov2025_articles_v2.10k', 2, 5), ('meta-llama/Llama-3.2-3B', 'guardian_from_nov2025_articles_v2.10k', 3, 5), ('meta-llama/Llama-3.2-3B', 'guardian_from_nov2025_articles_v2.10k', 4, 5), ('meta-llama/Llama-3.2-3B', 'guardian_from_nov2025_articles_v2.10k', 5, 5), ('meta-llama/Llama-3.2-3B', 'Hello-SimpleAI/HC3', 1, 5), ('meta-llama/Llama-3.2-3B', 'Hello-SimpleAI/HC3', 2, 5), ('meta-llama/Llama-3.2-3B', 'Hello-SimpleAI/HC3', 3, 5), ('meta-llama/Llama-3.2-3B', 'Hello-SimpleAI/HC3', 4, 5), ('meta-llama/Llama-3.2-3B', 'Hello-SimpleAI/HC3', 5, 5)]
Number of combinations: 10
Paths: [['model_arguments', 'model_id'], ['data_arguments', 'input_version'], ['steering_arguments', 'noise_offset'], ['steering_arguments', 'noise_max']]
Saving variant to: ../params/Small_llm_rerun__meta-llama_Llama-3.2-3B_guardian_from_nov2025_articles_v2.10k_1_5.yaml
Saving variant to: ../

In [26]:
# create a bash script that runs all the newly creates params files
bash_text = ""
for filename in file_names:
    bash_text += f"python {python_script} {filename[1:]}\n"

with open(f"../{output_param_name}.sh", 'w') as f:
    f.write(bash_text)


In [27]:
print(len(param_list))

10


In [28]:
pprint.pprint(param_list[-1])

{'comparison_arguments': {'compared_text_type': 'steered_bis'},
 'data_arguments': {'input_version': 'Hello-SimpleAI/HC3',
                    'max_loaded_samples': 1000,
                    'truncate_input_words': 6},
 'detection_arguments': {'lda_parameters': {'shrinkage': None,
                                            'solver': 'lsqr'},
                         'max_seq_length': 512,
                         'mlp_parameters': {'batch_size': 512,
                                            'hidden_dims': [2048, 64, 64, 32],
                                            'learning_rate': 0.001,
                                            'num_epochs': 1},
                         'model_type': 'mlp',
                         'number_of_bits': 2,
                         'number_prompts_truncation': None,
                         'seed': 1,
                         'sentence_array': False,
                         'token_aggregation': False,
                         'transformer_parame